# 小线性头对数据噪声清洗

## 模型和数据构建

本版简化代码并完全采用feature cache，即预提取feature之后直接保存为pt

### 1. 环境准备和Helper

In [2]:
import httpx
#from bs4 import BeautifulSoup
import re
import json
import pandas as pd
import os
import random
import time
import openai
import requests
import importlib
import torch
import pathlib
from pathlib import Path
import yaml
import sqlite3
from PIL import Image, ImageOps
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection,pipeline,AutoModel
from accelerate import Accelerator
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoImageProcessor
from transformers.image_utils import load_image
load_dotenv(override=False)
from huggingface_hub import login
login(token=os.getenv("HF_TOKEN"))
current_dir = os.getcwd()
PROJECT_ROOT = os.path.dirname(os.path.dirname(current_dir))
db_path = os.path.join(PROJECT_ROOT, "data", "commons_image_manifest.sqlite")

def load_img_with_orientation(path):
    img = Image.open(path).convert("RGB")
    img = ImageOps.exif_transpose(img)  # 确保方向正确
    return img

OPENAI_MODEL_NAME = "gpt-5.4-mini"
LOCAL_VLM_NAME = "qwen/qwen3.6-27b"
LM_STUDIO_ENDPOINT = "http://127.0.0.1:1234"
PROJECT_ROOT

def import_pipeline_utils(project_root: Path):
    utils_path = project_root / "pipeline" / "utils.py"
    spec = importlib.util.spec_from_file_location("wakareeru_pipeline_utils", utils_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load utils.py from {utils_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

utils = import_pipeline_utils(Path(PROJECT_ROOT))

print(f" Metal Availablility: {torch.backends.mps.is_available()}\nCuda Availability: {torch.cuda.is_available()}\nDevice count: {torch.cuda.device_count()}")
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")
# 从pipeline_config读取noise detection粒度配置
# 之后label代表选定的粒度标签。
try:
    with open(Path(PROJECT_ROOT) / "config" / "pipeline_config.yaml", "r", encoding="utf-8") as _f:
        _pipeline_cfg = yaml.safe_load(_f)
    NOISE_LABEL_GRANULARITY: str = _pipeline_cfg.get("noise_detection", {}).get("label_granularity", "fine_grained_series")
except Exception:
    NOISE_LABEL_GRANULARITY = "fine_grained_series"




Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


 Metal Availablility: True
Cuda Availability: False
Device count: 0
Using device: mps


In [3]:
from pathlib import Path
from typing import Callable

import numpy as np
import yaml

# Demo阶段的决策：
# - 先在线裁剪：bbox、padding、resize策略还在试，迭代最快，也不重复占磁盘。
# - 策略稳定后再离线缓存：多epoch反复训练同一批crop时再用materialize_crops。
# - batch里按原图路径复用已打开图片，避免同一原图的多个crop重复IO。
DATA_ROOT = Path(PROJECT_ROOT) / "data"
IMG_ROOT = DATA_ROOT
CROP_CACHE_ROOT = DATA_ROOT / "crops_demo"


def load_crop_manifest(
    db_path=db_path,
    series: list[str] | None = None,
    power_type: str | None = None,
    crop_status: str | None = None,
    min_score: float | None = None,
    limit: int | None = None,
    shuffle: bool = False,
    seed: int = 42,
    label_granularity: str = NOISE_LABEL_GRANULARITY,
) -> pd.DataFrame:
    """从SQLite读取crop元数据，并拼上本地原图路径。

    结果DataFrame包含 ``label`` 列，内容由 ``label_granularity`` 决定：
    - ``"series"``            : 直接复制 series
    - ``"fine_grained_series"``: 优先用 fine_grained_series，NULL时回落到 series
    - ``"submodel"``          : 优先用 submodel，NULL时依次回落到 fine_grained_series、series
    """
    where = ["i.downloaded_path IS NOT NULL", "i.downloaded_path != ''"]
    params: list[object] = []

    if series:
        where.append("c.series IN (" + ",".join(["?"] * len(series)) + ")")
        params.extend(series)
    if power_type is not None:
        where.append("c.power_type = ?")
        params.append(power_type)
    if crop_status is not None:
        where.append("c.crop_status = ?")
        params.append(crop_status)
    if min_score is not None:
        where.append("COALESCE(c.detector_score, 0) >= ?")
        params.append(min_score)

    sql_limit = None if shuffle else limit
    limit_sql = "LIMIT ?" if sql_limit is not None else ""
    if sql_limit is not None:
        params.append(limit)

    sql = f"""
    SELECT
        c.id AS crop_id,
        c.image_id,
        c.crop_index,
        c.series,
        c.power_type,
        c.detector_label,
        c.detector_score,
        c.box_x1, c.box_y1, c.box_x2, c.box_y2,
        i.downloaded_path,
        i.file_title,
        i.width AS image_width,
        i.height AS image_height,
        i.fine_grained_series,
        i.submodel
    FROM crops c
    JOIN images i ON i.id = c.image_id
    WHERE {' AND '.join(where)}
    ORDER BY c.id
    {limit_sql}
    """
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(sql, conn, params=params)

    if shuffle and len(df):
        n = min(limit or len(df), len(df))
        df = df.sample(n=n, random_state=seed).reset_index(drop=True)

    # 计算有效分类标签
    if label_granularity == "submodel":
        df["label"] = df["submodel"].fillna(df["fine_grained_series"]).fillna(df["series"])
    elif label_granularity == "fine_grained_series":
        df["label"] = df["fine_grained_series"].fillna(df["series"])
    else:
        df["label"] = df["series"]

    return df


In [4]:
def _source_image_path(row: pd.Series | dict, img_root: Path = IMG_ROOT) -> Path:
    path = Path(row["downloaded_path"])
    return path if path.is_absolute() else img_root / path


def _expanded_box(row: pd.Series | dict, image_size: tuple[int, int], pad_frac: float = 0.04):
    width, height = image_size
    x1, y1, x2, y2 = (float(row[k]) for k in ["box_x1", "box_y1", "box_x2", "box_y2"])
    pad = max(x2 - x1, y2 - y1) * pad_frac
    left = max(0, int(np.floor(x1 - pad)))
    top = max(0, int(np.floor(y1 - pad)))
    right = min(width, int(np.ceil(x2 + pad)))
    bottom = min(height, int(np.ceil(y2 + pad)))
    if right <= left or bottom <= top:
        raise ValueError(f"bad crop box for crop_id={row.get('crop_id', row.get('id'))}: {(left, top, right, bottom)}")
    return left, top, right, bottom


def crop_from_image(
    img: Image.Image,
    row: pd.Series | dict,
    pad_frac: float = 0.04,
) -> Image.Image:
    return img.crop(_expanded_box(row, img.size, pad_frac=pad_frac))


def load_crop(
    row: pd.Series | dict,
    img_root: Path = IMG_ROOT,
    pad_frac: float = 0.04,
) -> Image.Image:
    img = load_img_with_orientation(_source_image_path(row, img_root=img_root))
    return crop_from_image(img, row, pad_frac=pad_frac)


### 数据加载与Feature提取缓存

直接进行数据读取和feature cache，解耦之前没有必要的完整模型结构

In [5]:
# 初始化
config = utils.load_pipeline_config(Path(PROJECT_ROOT) / "config" / "pipeline_config.yaml")
noise_detection_cfg = config.get("noise_detection", {})
utils.init_db()

In [6]:
# 读取manifest

df_crops = load_crop_manifest(
    db_path=db_path,
    series= None if noise_detection_cfg.get("full_series") else noise_detection_cfg.get("series_test_scope"),
    label_granularity=NOISE_LABEL_GRANULARITY,
)
df_crops



,crop_id,image_id,crop_index,series,power_type,detector_label,detector_score,box_x1,box_y1,box_x2,box_y2,downloaded_path,file_title,image_width,image_height,fine_grained_series,submodel,label
0,1,2710,0,E6系,EMU,a train,0.652439,199.860168,357.479614,2665.539795,1947.217041,img/E6系/c77da1_E6系新幹線電車.JPG,File:E6系新幹線電車.JPG,2816,2112,E6系,NaN,E6系
1,2,947,0,E231系,EMU,a train,0.522514,62.674820,26.810646,5570.139160,3939.376953,img/E231系/db7b5f_East Japan Railway Series E23...,File:East Japan Railway Series E231-3000 Hachi...,6000,4000,E231系,E231-3000,E231系
2,3,3851,0,E233系,EMU,a train,0.748011,869.965698,551.989014,2554.307617,1738.222656,img/E233系/3043f4_TOKYO STATION JAPAN JUNE 2012...,File:TOKYO STATION JAPAN JUNE 2012 (7454198326...,3639,2011,E233系,E233,E233系
3,4,3485,0,E231系,EMU,a train,0.590797,430.508148,9.365750,3125.815918,2631.373047,img/E231系/f92f82_JREast-E231-500soubu.JPG,File:JREast-E231-500soubu.JPG,3648,2736,E231系,E231-500,E231系
4,5,3279,0,E657系,EMU,a train,0.890025,3962.527832,1791.068848,6265.960938,2912.862549,img/E657系/02495f_At Tokyo 2024 672.jpg,File:At Tokyo 2024 672.jpg,6960,4640,E657系,NaN,E657系
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5321,5865,12637,0,E217系,EMU,a train,0.578131,168.907776,22.230806,1084.034302,922.859375,img/E217系/d9c1ea_総武快速線E217系.JPG,File:総武快速線E217系.JPG,1280,960,E217系,NaN,E217系
5322,5866,15083,0,E231系,EMU,a train,0.612004,256.193604,376.524323,2905.977539,1723.626953,img/E231系/46b5f1_Toukyu5050 4110F.jpg,File:Toukyu5050 4110F.jpg,3008,2000,E231系,Tokyu 5050-4000,E231系
5323,5867,15407,0,E233系,EMU,a train,0.493209,11.403076,476.928314,2276.470703,2456.458984,img/E233系/49c627_Photo-with-Xiaomi15Ultra 264.jpg,File:Photo-with-Xiaomi15Ultra 264.jpg,4096,3072,E233系,E233-5000,E233系
5324,5868,15408,0,E233系,EMU,a train,0.640152,13.689087,152.174011,3762.171875,2996.280518,img/E233系/f413fb_Photo-with-Xiaomi15Ultra 265.jpg,File:Photo-with-Xiaomi15Ultra 265.jpg,4096,3072,E233系,E233-5000,E233系


In [7]:
pretrained_model_name = noise_detection_cfg["hf_model_name"]
processor = AutoImageProcessor.from_pretrained(pretrained_model_name)
model = AutoModel.from_pretrained(
    pretrained_model_name, 
    device_map="auto", 
)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

In [11]:
LABEL_COL = "label"
labels = sorted(df_crops[LABEL_COL].dropna().astype(str).unique())
label_to_id = {label: idx for idx, label in enumerate(labels)}
id_to_label = {idx: label for label, idx in label_to_id.items()}


class CropImageDataset(torch.utils.data.Dataset):
    def __init__(
            self,
            df_crops: pd.DataFrame,
            img_root: Path = IMG_ROOT,
            pad_frac: float = 0.04):
        self.df_crops = df_crops.reset_index(drop=True)
        self.img_root = img_root
        self.pad_frac = pad_frac
    
    def __len__(self):
        return len(self.df_crops)
    
    
    def __getitem__(self, idx):
        row = self.df_crops.iloc[idx]
        cropped_img = load_crop(row, img_root=self.img_root, pad_frac=self.pad_frac)
        return cropped_img, row.to_dict()

def collate_fn(batch):
    images, metas = zip(*batch)
    image_tensors = processor(images=list(images), return_tensors="pt")["pixel_values"]
    labels = torch.tensor([label_to_id[str(meta[LABEL_COL])] for meta in metas], dtype=torch.long)
    crop_ids = torch.tensor([int(meta["crop_id"]) for meta in metas], dtype=torch.long)
    return image_tensors, labels, crop_ids
    

In [8]:
embed_dim = model.config.hidden_size
embed_dim

384

In [12]:
device = next(model.parameters()).device
print(device)
image_tensors = image_tensors.to(device)
print(image_tensors.device)


mps:0


NameError: name 'image_tensors' is not defined

In [10]:
batch_size = noise_detection_cfg.get("feature_extraction_batch_size", 16)
dataset = CropImageDataset(df_crops)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, 
                                    shuffle=False, collate_fn=collate_fn, num_workers=0)

all_feature = []
all_labels = []
all_crop_ids = []

model.eval()
with torch.inference_mode():
    for batch in tqdm(dataloader):
        
        image_tensors, labels, crop_ids = batch
        image_tensors = image_tensors.to(model.device)
        outputs = model(image_tensors)
        
        pooled_output = outputs.pooler_output #CLS output
        
        all_feature.append(pooled_output.cpu())
        all_labels.append(labels)
        all_crop_ids.append(crop_ids)

feature_cache = {
    "features": torch.cat(all_feature, dim=0),
    "labels": torch.cat(all_labels, dim=0),
    "crop_ids": torch.cat(all_crop_ids, dim=0),
    "label_to_id": label_to_id,
    "id_to_label": id_to_label,
    "model_name": pretrained_model_name,
}

# saving

data_length = df_crops.shape[0]
class_num = len(label_to_id)
print(f"Extracted features for {data_length} crops, with {class_num} unique labels.")
file_name = f"demo_dinov3_features_{data_length}crops_{class_num}labels.pt"
torch.save(feature_cache, DATA_ROOT / "feature_cache" / file_name)

NameError: name 'CropImageDataset' is not defined

## 简单线性头训练并进行loss追踪

实现内容：
- 一个单层全连结层，\[embed_dim, class]维，直接以刚才的feature为输入。
- loss tracking, loss_fn处不进行reduction，记录完之后再手动mean。
- 用optim.lr_scheduler进行动态lr

In [13]:
from torch import nn
class LinearHead(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.linear = torch.nn.Linear(input_dim, num_classes)
    
    def forward(self, x):
        return self.linear(x)

lr_max = 1e-3
lr_min = 1e-5
period_steps = 10
epochs = 30
weight_decay = 1e-4

def make_sawtooth_lambda(lr_max: float, lr_min: float, period_steps: int):
    """
    Return a LambdaLR-compatible function.

    At step 0: lr = lr_max
    At step period_steps - 1: approximately lr_min
    At step period_steps: jumps back to lr_max
    """
    if period_steps <= 1:
        raise ValueError("period_steps must be > 1")
    if lr_max <= 0 or lr_min < 0:
        raise ValueError("lr_max must be > 0 and lr_min must be >= 0")
    if lr_min > lr_max:
        raise ValueError("lr_min must be <= lr_max")

    min_ratio = lr_min / lr_max

    def lr_lambda(step: int):
        t = step % period_steps
        progress = t / (period_steps - 1)  # 0 -> 1
        ratio = 1.0 - progress * (1.0 - min_ratio)
        return ratio

    return lr_lambda


train_device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
head = LinearHead(input_dim=embed_dim, num_classes=len(label_to_id)).to(train_device)
optimizer = torch.optim.AdamW(head.parameters(), lr=lr_max, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=make_sawtooth_lambda(lr_max, lr_min, period_steps),
)
criterion = torch.nn.CrossEntropyLoss(reduction='none')
print(f"Linear head training device: {train_device}")


Linear head training device: mps


In [15]:
## 加载数据
if 'file_name' not in locals():
    file_name = 'demo_dinov3_features_5326crops_23labels.pt'
feature_cache = torch.load(DATA_ROOT / "feature_cache" / file_name, map_location="cpu")
label_to_id = feature_cache['label_to_id']
id_to_label = feature_cache['id_to_label']

class FeatureDataset(torch.utils.data.Dataset):
    def __init__(self, feature_cache: dict):
        self.features = feature_cache["features"].float()
        self.labels = feature_cache["labels"].long()
        self.crop_ids = feature_cache["crop_ids"].long()

        assert len(self.features) == len(self.labels) == len(self.crop_ids)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx], self.crop_ids[idx]

feature_dataset = FeatureDataset(feature_cache)
head_batch_size = noise_detection_cfg.get("linear_head_train_batch_size", 32)
feature_dataloader = torch.utils.data.DataLoader(feature_dataset, 
                                            batch_size=head_batch_size, shuffle=True)


In [16]:
# 定义数据loss_record = (epoch, crop_id, label_id, pred_id, pred_confidence, correct, loss_value)
loss_records = []
epoch_records = []
global_step = 0
for epoch in tqdm(range(epochs), desc="Training epochs"):
    head.train()
    
    running_loss = 0.0
    running_correct = 0
    running_n = 0
    
    
    for x_cpu, y_cpu, crop_ids_cpu in feature_dataloader:
        x_dev = x_cpu.to(train_device)
        y_dev = y_cpu.to(train_device)
        
        optimizer.zero_grad(set_to_none=True)
        logits_dev = head(x_dev)
        loss_values_dev = criterion(logits_dev, y_dev)
        loss_dev = loss_values_dev.mean()
        loss_dev.backward()
        optimizer.step()
        scheduler.step()
        
        
        # 按batch记录全程loss，预测情况，正确性，softmax后的置信度
        with torch.no_grad():
            probs_dev = torch.softmax(logits_dev, dim=1)
            pred_confidence_dev, predict_dev = probs_dev.max(dim=1)
            correct_dev = predict_dev.eq(y_dev)
        
        loss_values_cpu = loss_values_dev.detach().cpu()
        predict_cpu = predict_dev.detach().cpu()
        pred_confidence_cpu = pred_confidence_dev.detach().cpu()
        correct_cpu = correct_dev.detach().cpu()
        y_cpu = y_dev.detach().cpu()
        crop_ids_cpu = crop_ids_cpu.cpu()
        
        #写入tuple记录
        for i, cropid in enumerate(crop_ids_cpu.tolist()):
            loss_records.append(
                (   epoch, 
                    cropid, 
                    y_cpu[i].item(), 
                    predict_cpu[i].item(), 
                    pred_confidence_cpu[i].item(), 
                    correct_cpu[i].item(), 
                    loss_values_cpu[i].item())
            )

        running_loss += float(loss_values_cpu.sum())
        running_correct += int(correct_cpu.sum())
        running_n += int(y_cpu.numel())
        global_step += 1
    # epoch信息记录
    epoch_records.append({
        "epoch": epoch,
        "loss": running_loss / max(1, running_n),
        "accuracy": running_correct / max(1, running_n),
        "n": running_n,
    })
    
loss_history = pd.DataFrame(
    loss_records,
    columns=["epoch", "crop_id", "label_id", "pred_id", "pred_confidence", "correct", "loss_value"],
)
epoch_history = pd.DataFrame(epoch_records)

Training epochs:   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
loss_history

## 简单loss mean可视化

只按每个crop的平均loss排序，快速查看最可疑样本。

In [ ]:
def plot_loss_curve(loss_history: pd.DataFrame, epoch_history: pd.DataFrame | None = None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    batch_loss = loss_history.groupby(["epoch"], as_index=False).agg(loss_mean=("loss_value", "mean"))
    axes[0].plot(batch_loss["epoch"], batch_loss["loss_mean"], marker="o")
    axes[0].set_title("Mean loss by epoch")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")

    if epoch_history is not None and len(epoch_history):
        axes[1].plot(epoch_history["epoch"], epoch_history["accuracy"], marker="o", color="tab:green")
        axes[1].set_title("Accuracy by epoch")
        axes[1].set_xlabel("epoch")
        axes[1].set_ylabel("accuracy")
    else:
        axes[1].axis("off")

    plt.tight_layout()
    return fig


def summarize_loss_mean(loss_history: pd.DataFrame, rows: pd.DataFrame) -> pd.DataFrame:
    stats = loss_history.groupby("crop_id", as_index=False).agg(
        loss_mean=("loss_value", "mean"),
        seen=("loss_value", "size"),
        error_rate=("correct", lambda x: 1.0 - float(np.mean(x))),
        pred_confidence_mean=("pred_confidence", "mean"),
    )

    meta_cols = [
        "crop_id", "image_id", "crop_index", "series", "label", "power_type",
        "detector_label", "detector_score", "file_title", "downloaded_path",
        "box_x1", "box_y1", "box_x2", "box_y2",
    ]
    meta = rows[[c for c in meta_cols if c in rows.columns]].drop_duplicates("crop_id")
    stats = stats.merge(meta, on="crop_id", how="left")
    return stats.sort_values("loss_mean", ascending=False).reset_index(drop=True)


def show_top_loss_mean_crops(
    loss_stats: pd.DataFrame,
    top_k: int = 24,
    pad_frac: float = 0.04,
    cols: int = 6,
):
    top = loss_stats.head(top_k).reset_index(drop=True)
    rows_n = int(np.ceil(len(top) / cols))
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 3.0, rows_n * 3.6))
    axes = np.array(axes).reshape(-1)

    for ax in axes[len(top):]:
        ax.axis("off")

    for i, row in top.iterrows():
        ax = axes[i]
        try:
            crop = load_crop(row, pad_frac=pad_frac)
            ax.imshow(crop)
            title = (
                f"#{i + 1} crop {int(row['crop_id'])}\n"
                f"loss_mean={row['loss_mean']:.2f} err={row['error_rate']:.2f}\n"
                f"{row.get('label', row.get('series', ''))}"
            )
        except Exception as exc:
            title = f"load failed: crop={row.get('crop_id')}\n{type(exc).__name__}: {exc}"
        ax.set_title(title, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    display_cols = ["crop_id", "label", "series", "loss_mean", "error_rate", "pred_confidence_mean", "file_title"]
    display(top[[c for c in display_cols if c in top.columns]])
    return fig, top


In [ ]:
plot_loss_curve(loss_history, epoch_history)
loss_mean_stats = summarize_loss_mean(loss_history, df_crops)
show_top_loss_mean_crops(loss_mean_stats, top_k=24)
